# Example of how to use the unfairness reduction proposed by PUFFLE in a centralised learning context.

Despite PUFFLE being originally designed for federated learning, it can also be applied in a centralised learning context. 
This example demonstrates how to use the unfairness reduction proposed by PUFFLE in a centralised learning context.

We will use the `Dutch Census` dataset and the `PUFFLE` library to train a model that reduces unfairness in the predictions.

In [1]:
from puffle.FairReg.Regularization import RegularizationLoss
from puffle.FairReg.Utils.metric import compute_demographic_disparity
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import random
import os
from puffle.FairModel.fair_model import PUFFLEModel


## Utility Functions

In [2]:
def prepare_dutch(df: pd.DataFrame, sensitive_attribute: str, target: str):
    feature_columns = [
        "age",
        "household_position",
        "household_size",
        "prev_residence_place",
        "citizenship",
        "country_birth",
        "edu_level",
        "economic_status",
        "cur_eco_activity",
        "Marital_status",
        "sex_binary",
    ]

    scaler = MinMaxScaler()
    dutch_df = scaler.fit_transform(df[feature_columns])
    dutch_df = pd.DataFrame(dutch_df, columns=feature_columns)

    # In this case we know that the sensitive attribute is a binary column
    sensitive_attribute = df[f"{sensitive_attribute}"].values
    target = df[f"{target}"].values

    # train-test split
    X_train, X_test, y_train, y_test, sensitive_train, sensitive_test = train_test_split(
        dutch_df,
        target,
        sensitive_attribute,
        test_size=0.2,
        random_state=42,
        stratify=sensitive_attribute,
    )

    # convert to numpy arrays
    X_train = np.array(X_train)
    X_test = np.array(X_test)
    y_train = np.array(y_train)
    y_test = np.array(y_test)
    sensitive_train = np.array(sensitive_train)
    sensitive_test = np.array(sensitive_test)

    return X_train, X_test, y_train, y_test, sensitive_train, sensitive_test


class TabularDataset(Dataset):
    def __init__(self, x, z, y):
        """
        Initialize the custom dataset with x (features), z (sensitive values), and y (targets).

        Args:
        x (list of tensors): List of input feature tensors.
        z (list): List of sensitive values.
        y (list): List of target values.
        """
        self.samples = x
        self.sensitive_features = z
        self.targets = y
        self.indexes = range(len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        """
        Get a single data point from the dataset.

        Args:
        idx (int): Index to retrieve the data point.

        Returns:
        sample (dict): A dictionary containing 'x', 'z', and 'y'.
        """
        x_sample = self.samples[idx]
        z_sample = self.sensitive_features[idx]
        y_sample = self.targets[idx]

        return x_sample, z_sample, y_sample, self.indexes[idx], idx


class LinearClassificationNet(nn.Module):
    """
    A fully-connected single-layer linear NN for classification.
    """

    def __init__(self, input_size=12, output_size=2):
        super(LinearClassificationNet, self).__init__()
        self.layer1 = nn.Linear(input_size, output_size, bias=False)

    def forward(self, x):
        x = self.layer1(x.float())
        return x

## Prepare dutch dataset

In [3]:
df = pd.read_csv("./dutch_census.csv")

X_train, X_test, y_train, y_test, sensitive_train, sensitive_test = prepare_dutch(
    df, sensitive_attribute="sex_binary", target="occupation_binary"
)

train_dataset = TabularDataset(
    x=np.hstack((X_train, np.ones((X_train.shape[0], 1)))).astype(np.float32),
    z=sensitive_train,
    y=y_train,
)

test_dataset = TabularDataset(
    x=np.hstack((X_test, np.ones((X_test.shape[0], 1)))).astype(np.float32),
    z=sensitive_test,
    y=y_test,
)

In [4]:
df.head()

,age,household_position,household_size,prev_residence_place,citizenship,country_birth,edu_level,economic_status,cur_eco_activity,Marital_status,sex_binary,occupation_binary
0,6,1131,112,1,1,1,5,111,135,1,1,0
1,10,1122,113,1,1,1,2,111,122,2,0,1
2,8,1122,113,1,1,1,2,111,122,2,1,0
3,12,1121,112,1,1,1,1,111,137,2,1,1
4,4,1110,114,1,1,1,2,111,138,1,0,1


In [5]:
compute_demographic_disparity(y=torch.tensor(sensitive_test), z=torch.tensor(y_test))

0.3057669997215271

## Train a simple model without any unfairness reduction or privacy mitigation

In [6]:
# Create the model that we will train, for Dutch we will use a LinearClassificationNet
# defined inside this Library in the Models/logistic_regression_net.py file
model = LinearClassificationNet()
batch_size = 333
lr = 0.019925917176300392
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
epochs = 5
seed = 42

# seed the model
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
os.environ["PYTHONHASHSEED"] = str(seed)

In [7]:
# Create data loaders for training and testing
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [8]:
# Define some hyperparameters
model = LinearClassificationNet()
batch_size = 333
lr = 0.019925917176300392
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
epochs = 5

In [9]:
# Create data loaders for training and testing
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [10]:
# We will use the PUFFLEModel that is a weapper around the torch model.
# In this case we won't use any unfairness mitigation
puffle_model = PUFFLEModel(
    model=model, 
    optimizer=optimizer,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

In [11]:
train_metrics = puffle_model.train(
    train_loader=train_loader,
    epochs=epochs,
)

Epoch 1/5, Train Loss: 0.5125, Train Acc: 0.7615, Train F1: 0.7612, Train Disparity: 0.5746
Epoch 2/5, Train Loss: 0.4303, Train Acc: 0.8223, Train F1: 0.8218, Train Disparity: 0.3490
Epoch 3/5, Train Loss: 0.4197, Train Acc: 0.8240, Train F1: 0.8235, Train Disparity: 0.3304
Epoch 4/5, Train Loss: 0.4186, Train Acc: 0.8238, Train F1: 0.8232, Train Disparity: 0.3270
Epoch 5/5, Train Loss: 0.4164, Train Acc: 0.8238, Train F1: 0.8232, Train Disparity: 0.3254


In [12]:
puffle_model.evaluate(
    data_loader=test_loader,
)

{'loss': 0.40397857008753596,
 'accuracy': 0.8313472360145647,
 'f1': 0.8307563723058641,
 'disparity': 0.33450549840927124}

## Train with PUFFLE unfairness reduction